In [0]:
%python
from pyspark.sql import functions as F

CATALOG = "streaming_sentiment_intelligence"
SCHEMA = "default"

SOURCE_PATH = "/Volumes/streaming_sentiment_intelligence/default/raw/"

STREAM_BRONZE = f"{CATALOG}.{SCHEMA}.streaming_bronze_reviews"
STREAM_SILVER = f"{CATALOG}.{SCHEMA}.streaming_silver_reviews"
STREAM_GOLD = f"{CATALOG}.{SCHEMA}.streaming_gold_sentiment"

CHECKPOINT_BASE = "/Volumes/streaming_sentiment_intelligence/default/raw/checkpoints/"

print("Streaming pipeline configuration loaded.")

Streaming pipeline configuration loaded.


In [0]:
# ============================================================
# 2. AUTO LOADER - STREAMING INGESTION
# ============================================================

streaming_bronze_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", CHECKPOINT_BASE + "schema")
    .option("cloudFiles.inferColumnTypes", "true")
    .option("header", "true")
    .load(SOURCE_PATH)
)

print("Auto Loader configured successfully.")

Auto Loader configured successfully.


In [0]:
# ============================================================
# 3. ADD STREAMING METADATA
# ============================================================

streaming_bronze_df = (
    streaming_bronze_df
    .withColumnRenamed("Unnamed: 0", "id")
    .withColumn(
        "_stream_ingestion_timestamp",
        F.current_timestamp()
    )
    .withColumn(
        "_source_file",
        F.col("_metadata.file_path")
    )
)

In [0]:
# ============================================================
# 4. STREAMING BRONZE
# ============================================================

bronze_query = (
    streaming_bronze_df.writeStream
    .format("delta")
    .outputMode("append")
    .option(
        "checkpointLocation",
        CHECKPOINT_BASE + "bronze"
    )
    .trigger(availableNow=True)
    .toTable(STREAM_BRONZE)
)

print("✅ Streaming Bronze pipeline completed.")

✅ Streaming Bronze pipeline completed.


In [0]:
# ============================================================
# 5. READ STREAMING BRONZE
# ============================================================

stream_silver_df = (
    spark.readStream
    .table(STREAM_BRONZE)
)

print("Streaming Bronze connected to Silver.")

Streaming Bronze connected to Silver.


In [0]:
# ============================================================
# 6. STREAMING SILVER TRANSFORMATION
# ============================================================

stream_silver_df = (
    stream_silver_df

    # Clean text
    .withColumn(
        "review_text",
        F.trim(F.col("Text"))
    )

    # Clean sentiment
    .withColumn(
        "sentiment",
        F.initcap(F.trim(F.col("Sentiment")))
    )

    # Clean user
    .withColumn(
        "user",
        F.trim(F.col("User"))
    )

    # Clean platform
    .withColumn(
        "platform",
        F.trim(F.col("Platform"))
    )

    # Clean country
    .withColumn(
        "country",
        F.trim(F.col("Country"))
    )

    # Numeric conversions
    .withColumn(
        "likes",
        F.col("Likes").cast("double")
    )

    .withColumn(
        "retweets",
        F.col("Retweets").cast("double")
    )

    # Timestamp
    .withColumn(
        "timestamp",
        F.to_timestamp(F.col("Timestamp"))
    )

    # NLP features
    .withColumn(
        "text_length",
        F.length(F.col("review_text"))
    )

    .withColumn(
        "word_count",
        F.size(
            F.split(
                F.trim(F.col("review_text")),
                r"\s+"
            )
        )
    )

    # Engagement
    .withColumn(
        "engagement_score",
        F.coalesce(F.col("Likes").cast("double"), F.lit(0))
        +
        F.coalesce(F.col("Retweets").cast("double"), F.lit(0))
    )

    # Processing timestamp
    .withColumn(
        "_silver_processing_timestamp",
        F.current_timestamp()
    )
)

In [0]:
# ============================================================
# 7. STREAMING DATA QUALITY
# ============================================================

stream_silver_df = stream_silver_df.filter(
    F.col("review_text").isNotNull()
)

stream_silver_df = stream_silver_df.filter(
    F.length(F.trim(F.col("review_text"))) > 0
)

stream_silver_df = stream_silver_df.filter(
    F.col("sentiment").isNotNull()
)

In [0]:
# ============================================================
# 8. STREAMING SILVER
# ============================================================

silver_query = (
    stream_silver_df.writeStream
    .format("delta")
    .outputMode("append")
    .option(
        "checkpointLocation",
        CHECKPOINT_BASE + "silver"
    )
    .trigger(availableNow=True)
    .toTable(STREAM_SILVER)
)

print("✅ Streaming Silver pipeline completed.")

✅ Streaming Silver pipeline completed.


In [0]:
# ============================================================
# 9. STREAMING GOLD
# ============================================================

stream_gold_df = (
    spark.readStream
    .table(STREAM_SILVER)
    .groupBy("sentiment")
    .agg(
        F.count("*").alias("review_count"),
        F.round(F.avg("likes"), 2).alias("avg_likes"),
        F.round(F.avg("retweets"), 2).alias("avg_retweets"),
        F.round(F.avg("engagement_score"), 2).alias("avg_engagement"),
        F.round(F.avg("text_length"), 2).alias("avg_text_length"),
        F.round(F.avg("word_count"), 2).alias("avg_word_count")
    )
)

In [0]:
# ============================================================
# 10. STREAMING GOLD TABLE
# ============================================================

gold_query = (
    stream_gold_df.writeStream
    .format("delta")
    .outputMode("complete")
    .option(
        "checkpointLocation",
        CHECKPOINT_BASE + "gold"
    )
    .trigger(availableNow=True)
    .toTable(STREAM_GOLD)
)

print("✅ Streaming Gold pipeline completed.")

✅ Streaming Gold pipeline completed.


In [0]:
# ============================================================
# 11. VERIFY STREAMING TABLES
# ============================================================

print("Streaming Bronze:")
display(
    spark.table(STREAM_BRONZE).limit(10)
)

print("Streaming Silver:")
display(
    spark.table(STREAM_SILVER).limit(10)
)

print("Streaming Gold:")
display(
    spark.table(STREAM_GOLD)
)

Streaming Bronze:


id,Text,Sentiment,Timestamp,User,Platform,Hashtags,Retweets,Likes,Country,Year,Month,Day,Hour,_rescued_data,_stream_ingestion_timestamp,_source_file


Streaming Silver:


id,Text,sentiment,timestamp,user,platform,Hashtags,retweets,likes,country,Year,Month,Day,Hour,_rescued_data,_stream_ingestion_timestamp,_source_file,review_text,text_length,word_count,engagement_score,_silver_processing_timestamp


Streaming Gold:


sentiment,review_count,avg_likes,avg_retweets,avg_engagement,avg_text_length,avg_word_count


In [0]:
%sql
SELECT *
FROM streaming_sentiment_intelligence.default.streaming_gold_sentiment
ORDER BY review_count DESC;

sentiment,review_count,avg_likes,avg_retweets,avg_engagement,avg_text_length,avg_word_count
